## Profiling and Benchmarking XModalix
XModalix is rather slow compared to Varix, we assume this is because of the MultiModalDataSet and the CustomSampler. 
Before improving the code we profile and benchmark the components of the XModalix pipeline. We think the following modules could be relevant:
- MultiModalDataset
  - which is comprised of NumericDataset, or ImageDataset
- CoverageEnsuringSampler
- XModlixTrainer
  - which calls multiple GeneralTrainers
    - which is a child of BaseTrainer


We should also profile for four different cases:
  - single cell vs standard tabular
  - paired vs unpaired
  - image vs standard tabulas
  - image vs single cell

In [1]:
import os

p = os.getcwd()
d = "autoencodix_package"
if d not in p:
    raise FileNotFoundError(f"'{d}' not found in path: {p}")
os.chdir(os.sep.join(p.split(os.sep)[: p.split(os.sep).index(d) + 1]))
print(f"Changed to: {os.getcwd()}")


Changed to: /Users/maximilianjoas/development/autoencodix_package


In [2]:
import torch
import numpy as np
import pandas as pd
from torch import nn
from torch.profiler import profile, ProfilerActivity, record_function
import autoencodix as acx
from autoencodix.trainers import _xmodal_trainer, _general_trainer
from autoencodix.base import BaseTrainer, BaseDataset
from autoencodix.base._base_dataset import DataSetTypes
from autoencodix.data import NumericDataset, MultiModalDataset, ImageDataset
from autoencodix.data._multimodal_dataset import CoverageEnsuringSampler
from autoencodix.utils.example_data import EXAMPLE_MULTI_SC, EXAMPLE_MULTI_BULK
import torch.utils.benchmark as benchmark



In [3]:
rna_file = os.path.join("data/XModalix-Tut-data/combined_rnaseq_formatted.parquet")
img_root = os.path.join("data/XModalix-Tut-data/images/tcga_fake")

#### Run XModalix to get access to all attributes we want to benchmark

In [4]:
from autoencodix.configs.xmodalix_config import XModalixConfig
from autoencodix.configs.default_config import DataConfig, DataInfo, DataCase
from autoencodix.modeling._imgfast_architecture import ImageVAEFastArchitecture
from autoencodix.modeling._varix_architecture import VarixArchitecture

clin_file = os.path.join("./data/XModalix-Tut-data/combined_clin_formatted.parquet")
rna_file = os.path.join("data/XModalix-Tut-data/combined_rnaseq_formatted.parquet")
img_root = os.path.join("data/XModalix-Tut-data/images/tcga_fake")

xmodalix_config = XModalixConfig(
    checkpoint_interval=100,
    class_param="CANCER_TYPE",
    epochs=1,
    beta=0.1,
    gamma=10,
    delta_class=100,
    delta_pair=300,
    latent_dim=6,
    k_filter=1000,
    batch_size=512,
    profiling=False,
    learning_rate=0.0005,
    requires_paired=False,
    device="cpu",
    loss_reduction="sum",
    data_case=DataCase.IMG_TO_IMG,
    data_config=DataConfig(
        data_info={
            "img": DataInfo(
                file_path=img_root,
                img_height_resize=32,
                img_width_resize=32,
                data_type="IMG",
                scaling="STANDARD",
                translate_direction="to",
                pretrain_epochs=0,),
            "img2": DataInfo(
                file_path=img_root,
                img_height_resize=32,
                img_width_resize=32,
                data_type="IMG",
                scaling="STANDARD",
                translate_direction="from",
                pretrain_epochs=0,),



            # "rna": DataInfo(
            #     file_path=rna_file,
            #     data_type="NUMERIC",
            #     scaling="STANDARD",
            #     pretrain_epochs=0,
            #     translate_direction="from",
            # ),
            # "rna2": DataInfo(
            #     file_path=rna_file,
            #     data_type="NUMERIC",
            #     scaling="STANDARD",
            #     pretrain_epochs=0,
            #     translate_direction="to",
            # ),
 
            "anno": DataInfo(file_path=clin_file, data_type="ANNOTATION", sep="\t"),
        },
        annotation_columns=["CANCER_TYPE_ACRONYM"],
    ),
)

xmodalix = acx.XModalix(config=xmodalix_config)#, model_map={DataSetTypes.NUM: VarixArchitecture, DataSetTypes.IMG: ImageVAEFastArchitecture})#, model_type=ImageVAEArchitecture)
result = xmodalix.run()


Given image size is possible, rescaling images to: 32x32
Successfully loaded 3230 images for img
Given image size is possible, rescaling images to: 32x32
Successfully loaded 3230 images for img2
calling normalize image in _process_ing_to_img_case
anno key: img
anno key: img2
Converting 2261 images to torch.float32 tensors...
Converting 2261 images to torch.float32 tensors...
Converting 646 images to torch.float32 tensors...
Converting 646 images to torch.float32 tensors...
Converting 323 images to torch.float32 tensors...
Converting 323 images to torch.float32 tensors...
key: train, type: <class 'dict'>
key: valid, type: <class 'dict'>
key: test, type: <class 'dict'>
Check if we need to pretrain: img.img
pretrain epochs : 0
No pretraining for img.img
Check if we need to pretrain: img.img2
pretrain epochs : 0
No pretraining for img.img2
--- Epoch 1/1 ---
split: train, n_samples: 2048.0
Epoch 1/1 - Train Loss: 3832.9915
Sub-losses - adver_loss: 14.4597, aggregated_sub_losses: 2994.1772, 

#### Getting attributes I want to profiler

In [ ]:
jds = result.datasets.train
ds.datasets['multi_bulk.rna'] = ds.datasets["img.img"]
xmodalix.config.skip_preprocessing = True
res = xmodalix.run(data=ds)

In [5]:
from autoencodix.data._multimodal_dataset import create_multimodal_collate_fn, CoverageEnsuringSampler
model = result.model
forward_fn = xmodalix._trainer._modalities_forward
loader = xmodalix._trainer._trainloader
dataset: MultiModalDataset = CoverageEnsuringSampler(multimodal_dataset=loader.dataset, batch_size=xmodalix_config.batch_size)
sampler = xmodalix._trainer
collate_fn = create_multimodal_collate_fn(multimodal_dataset=dataset)


In [6]:
activities = [ProfilerActivity.CPU]
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
    activities += [ProfilerActivity.CUDA]


sort_by_keyword = device + "_time_total"


with profile(activities=activities, record_shapes=True, profile_memory=True, with_stack=False,
    experimental_config=torch._C._profiler._ExperimentalConfig(verbose=True)) as prof:
    with record_function("model_inference"):
        xmodalix.fit()
        #forward_fn(next(iter(loader)))

print(prof.key_averages().table(sort_by=sort_by_keyword, row_limit=10))

STAGE:2025-12-15 09:42:32 97851:2919998 ActivityProfilerController.cpp:314] Completed Stage: Warm Up


Check if we need to pretrain: img.img
pretrain epochs : 0
No pretraining for img.img
Check if we need to pretrain: img.img2
pretrain epochs : 0
No pretraining for img.img2
--- Epoch 1/1 ---
split: train, n_samples: 2048.0
Epoch 1/1 - Train Loss: 3483.8237
Sub-losses - adver_loss: 15.8159, aggregated_sub_losses: 2664.8103, paired_loss: 474.4739, class_loss: 328.7238, img.img.recon_loss: 1341.8022, img.img.var_loss: 0.0000, img.img.loss: 1341.8022, img.img2.recon_loss: 1323.0081, img.img2.var_loss: 0.0000, img.img2.loss: 1323.0081, clf_loss: 1.4949
split: valid, n_samples: 323
Epoch 1/1 - Valid Loss: 2669.3723
Sub-losses - adver_loss: 14.3643, aggregated_sub_losses: 2047.1293, paired_loss: 359.6538, class_loss: 248.2248, img.img.recon_loss: 1023.3308, img.img.var_loss: 0.0000, img.img.loss: 1023.3308, img.img2.recon_loss: 1023.7984, img.img2.var_loss: 0.0000, img.img2.loss: 1023.7984, clf_loss: 1.4355
Storing checkpoint for epoch 0...


STAGE:2025-12-15 09:42:38 97851:2919998 ActivityProfilerController.cpp:320] Completed Stage: Collection
STAGE:2025-12-15 09:42:38 97851:2919998 ActivityProfilerController.cpp:324] Completed Stage: Post Processing


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                        model_inference         1.21%      58.974ms       100.00%        4.867s        4.867s      31.77 Mb    -164.90 Mb             1  
                                      aten::convolution         0.02%     777.000us        57.58%        2.802s      28.024ms     298.69 Mb           0 b           100  
                                     aten::_convolution         0.02%       1.038ms        57.57%        2.802s      28.016ms     298.69 Mb         -6

In [7]:
print(prof.key_averages(group_by_stack_n=5).table(sort_by=sort_by_keyword, row_limit=2))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                        model_inference         1.21%      58.974ms       100.00%        4.867s        4.867s      31.77 Mb    -164.90 Mb             1  
                                      aten::convolution         0.02%     777.000us        57.58%        2.802s      28.024ms     298.69 Mb           0 b           100  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ---------

In [8]:
for evt in prof.key_averages():
    print(evt)

<FunctionEventAvg key=model_inference self_cpu_time=58.974ms cpu_time=4.867s  self_cuda_time=0.000us cuda_time=0.000us input_shapes= cpu_memory_usage=33309944 cuda_memory_usage=0>
<FunctionEventAvg key=aten::empty self_cpu_time=1.181ms cpu_time=0.645us  self_cuda_time=0.000us cuda_time=0.000us input_shapes= cpu_memory_usage=800920352 cuda_memory_usage=0>
<FunctionEventAvg key=aten::detach self_cpu_time=215.000us cpu_time=0.533us  self_cuda_time=0.000us cuda_time=0.000us input_shapes= cpu_memory_usage=3656 cuda_memory_usage=0>
<FunctionEventAvg key=detach self_cpu_time=286.000us cpu_time=0.345us  self_cuda_time=0.000us cuda_time=0.000us input_shapes= cpu_memory_usage=2096 cuda_memory_usage=0>
<FunctionEventAvg key=aten::uniform_ self_cpu_time=3.713ms cpu_time=71.404us  self_cuda_time=0.000us cuda_time=0.000us input_shapes= cpu_memory_usage=0 cuda_memory_usage=0>
<FunctionEventAvg key=aten::fill_ self_cpu_time=21.669ms cpu_time=46.006us  self_cuda_time=0.000us cuda_time=0.000us input_sha

In [10]:
import csv
from pathlib import Path
csv_path = Path("test_profiling.csv")
with csv_path.open(mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(
        [
            "name",
            "cpu_time"
            "cuda_time"
            "cpu_memory_usage",
            "cuda_memory_usage",
            "input_shapes",
            "calls",
        ]
    )


    for evt in prof.key_averages():
                writer.writerow(
                    [
                        evt.key,
                        evt.cpu_time,
                        evt.cuda_time,
                        evt.cpu_memory_usage,
                        evt.cuda_memory_usage,
                        str(evt.input_shapes),
                        evt.count,
                    ]
                )


In [14]:
xmodalix._trainer._modality_dynamics["img.img"]["model"].mark_forward_method("decode")

In [3]:
from autoencodix.configs.xmodalix_config import XModalixConfig
from autoencodix.configs.default_config import DataConfig, DataInfo, DataCase
from autoencodix.modeling._imgfast_architecture import ImageVAEFastArchitecture
from autoencodix.modeling._varix_architecture import VarixArchitecture
rna_file = os.path.join("data/raw/scRNA_human_cortex.h5ad")

atac_file = os.path.join("data/raw/scATAC_human_cortex.h5ad")

xmodalix_config = XModalixConfig(
    checkpoint_interval=100,
    epochs=1,
    class_param="cell_type",
    beta=0.1,
    gamma=10,
    delta_class=100,
    delta_pair=300,
    latent_dim=6,
    k_filter=1000,
    batch_size=512,
    profiling=True,
    profile_logs="XM_SC_TCGA",
    learning_rate=0.0005,
    requires_paired=False,
    loss_reduction="sum",
    data_case=DataCase.SINGLE_CELL_TO_SINGLE_CELL,
    data_config=DataConfig(
        data_info={
            "atac": DataInfo(
                file_path=atac_file,
                data_type="NUMERIC",
                translate_direction="to",
                is_single_cell=True,
                pretrain_epochs=0,
            ),
            "rna": DataInfo(
                file_path=rna_file,
                data_type="NUMERIC",
                scaling="STANDARD",
                pretrain_epochs=0,
                translate_direction="from",
                is_single_cell=True,
            ),
        },
        annotation_columns=["cell_type"],
    ),
)

xmodalix = acx.XModalix(config=xmodalix_config)

result = xmodalix.run()
# del xmodalix
# del result


data in multi_sc: {'multi_sc': {'atac': AnnData object with n_obs × n_vars = 45549 × 19492
    obs: 'author_cell_type', 'age_group', 'donor_id', 'nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'TSS_percentile', 'nucleosome_signal', 'percent_mt', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'organism_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'is_primary_data', 'batch', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype'
    uns: 'batch_condition', 'schema_version', 'title'
    obsm: 'X_joint_wnn_umap', 'X_umap', 'rna': AnnData object with n_obs × n_vars = 45549 × 30113
    obs: 'author_cell_type', 'age_group', 'donor_id', 'nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


Processing train modality: rna
Processing valid split
Processing valid modality: atac
Processing valid modality: rna
Processing test split
Processing test modality: atac
Processing test modality: rna
key: train, type: <class 'dict'>
key: valid, type: <class 'dict'>
key: test, type: <class 'dict'>
Check if we need to pretrain: multi_sc.atac
pretrain epochs : 0
No pretraining for multi_sc.atac
Check if we need to pretrain: multi_sc.rna
pretrain epochs : 0
No pretraining for multi_sc.rna
--- Epoch 1/1 ---


STAGE:2025-12-15 15:43:12 16422:3361390 ActivityProfilerController.cpp:314] Completed Stage: Warm Up
[W CPUAllocator.cpp:249] Memory block of unknown size was allocated before the profiling started, profiler results will not include the deallocation event
STAGE:2025-12-15 15:43:17 16422:3361390 ActivityProfilerController.cpp:320] Completed Stage: Collection
STAGE:2025-12-15 15:43:17 16422:3361390 ActivityProfilerController.cpp:324] Completed Stage: Post Processing



PROFILER SUMMARY - Top Operations by CPU Time
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         1.19%      15.866ms        98.63%        1.311s     437.091ms       7.81 Mb      -7.81 Mb             3  
enumerate(DataLoader)#_SingleProcessDataLoaderIter._...        76.23%        1.013s        78.46%        1.043s     173.846ms      23.44 Mb           0 b             6  
                                             dataloader         1.14%      15.206ms        43.91%     5

data in multi_sc: {'multi_sc': {'atac': AnnData object with n_obs × n_vars = 45549 × 19492
    obs: 'author_cell_type', 'age_group', 'donor_id', 'nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'TSS_percentile', 'nucleosome_signal', 'percent_mt', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'organism_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'is_primary_data', 'batch', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype'
    uns: 'batch_condition', 'schema_version', 'title'
    obsm: 'X_joint_wnn_umap', 'X_umap', 'rna': AnnData object with n_obs × n_vars = 45549 × 30113
    obs: 'author_cell_type', 'age_group', 'donor_id', 'nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


Processing train modality: rna
Processing valid split
Processing valid modality: atac
Processing valid modality: rna
Processing test split
Processing test modality: atac
Processing test modality: rna
key: train, type: <class 'dict'>
key: valid, type: <class 'dict'>
key: test, type: <class 'dict'>
Check if we need to pretrain: multi_sc.atac
pretrain epochs : 0
No pretraining for multi_sc.atac
Check if we need to pretrain: multi_sc.rna
pretrain epochs : 0
No pretraining for multi_sc.rna
--- Epoch 1/1 ---


STAGE:2025-12-15 14:24:16 10161:3252023 ActivityProfilerController.cpp:314] Completed Stage: Warm Up
[W CPUAllocator.cpp:249] Memory block of unknown size was allocated before the profiling started, profiler results will not include the deallocation event
STAGE:2025-12-15 14:24:17 10161:3252023 ActivityProfilerController.cpp:320] Completed Stage: Collection
STAGE:2025-12-15 14:24:17 10161:3252023 ActivityProfilerController.cpp:324] Completed Stage: Post Processing



PROFILER SUMMARY - Top Operations by CPU Time
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         4.63%      23.510ms        95.63%     485.950ms     161.983ms       7.81 Mb      -7.81 Mb             3  
                                            total_batch        -0.03%    -152.000us        68.83%     349.766ms     116.589ms          56 b         -16 b             3  
                                     train_autoencoders        -0.34%   -1716.000us        47.89%     2